<a href="https://colab.research.google.com/github/chandraniraychowdhury5/DS/blob/main/Wind_Energy_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [8]:
df = pd.read_csv('/content/drive/MyDrive/datasetAndProgram/Location1.csv')
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df['Time'] = pd.to_datetime(df['Time'])
df.set_index('Time', inplace=True)
df.head(2)

,temperature_2m,relativehumidity_2m,dewpoint_2m,windspeed_10m,windspeed_100m,winddirection_10m,winddirection_100m,windgusts_10m,Power
Time,,,,,,,,,
2017-01-02 00:00:00,28.5,85,24.5,1.44,1.26,146,162,1.4,0.1635
2017-01-02 01:00:00,28.4,86,24.7,2.06,3.99,151,158,4.4,0.1424


Wind Shear Coefficient

Wind speed generally increases with height due to reduced surface friction.

The Wind Shear Coefficient quantifies this change and reflects atmospheric stability.

It helps estimate wind conditions across the turbine rotor sweep, improving power prediction accuracy.

In [13]:
df['Wind_Shear_Coefficient'] = np.log(df['windspeed_100m']/df['windspeed_10m'])/ np.log(100/10)

print(df['Wind_Shear_Coefficient'].describe())

df['Wind_Shear_Coefficient'] = df['Wind_Shear_Coefficient'].replace([np.inf, -np.inf], np.nan)

print(df['Wind_Shear_Coefficient'].describe())


count    4.380000e+04
mean              inf
std               NaN
min     -7.634280e-01
25%      1.714184e-01
50%      2.469580e-01
75%      3.086800e-01
max               inf
Name: Wind_Shear_Coefficient, dtype: float64
count    43797.000000
mean         0.244581
std          0.087942
min         -0.763428
25%          0.171415
50%          0.246955
75%          0.308663
max          1.466868
Name: Wind_Shear_Coefficient, dtype: float64


Air Density (ρ)

Wind turbines generate power from the kinetic energy of air, and air density directly affects the available energy.

Cold, dry air is denser than warm, humid air, allowing turbines to extract more energy at the same wind speed.

In [15]:
# temperatures to Celsius
temp_c = (df['temperature_2m'] - 32) * 5 / 9
dew_c = (df['dewpoint_2m'] - 32) * 5 / 9

P = df['surface_pressure_pa'] if 'surface_pressure_pa' in df.columns else 101325

# Saturation vapor pressure (Pa) using Bolton's formula
e = 6.112 * np.exp((17.67 * dew_c) / (dew_c + 243.5)) * 100

# Cap vapor pressure at total atmospheric pressure to prevent mathematical clipping errors
e = np.minimum(e, P)

# Gas constants & Temperature in Kelvin
Rd = 287.05
Rv = 461.495
T_k = temp_c + 273.15

# Air Density Calculation (kg/m³)
df['Air_Density'] = ((P - e) / (Rd * T_k)) + (e / (Rv * T_k))

print(df['Air_Density'].describe())


count    43800.000000
mean         1.249240
std          0.050790
min          1.136349
25%          1.205316
50%          1.249646
75%          1.289716
max          1.426598
Name: Air_Density, dtype: float64


Estimated Wind Speed at Hub Height

Estimating wind speed at the actual hub height provides a more realistic input for power prediction models.

In [19]:
hub_height = [10, 20, 30, 40, 50, 60, 70, 80, 90]
for h in hub_height:
    df[f'WindSpeed_HubHeight_{h}'] = (df['windspeed_100m'] * (h / 100) ** df['Wind_Shear_Coefficient'])
    print(df[f'WindSpeed_HubHeight_{h}'].describe())


count    43797.000000
mean         3.591393
std          1.649107
min          0.100000
25%          2.410000
50%          3.300000
75%          4.590000
max         13.450000
Name: WindSpeed_HubHeight_10, dtype: float64
count    43797.000000
mean         4.237200
std          1.886592
min          0.123202
25%          2.905383
50%          3.942945
75%          5.382379
max         15.302888
Name: WindSpeed_HubHeight_20, dtype: float64
count    43797.000000
mean         4.673279
std          2.050015
min          0.119236
25%          3.230786
50%          4.382389
75%          5.928696
max         16.502935
Name: WindSpeed_HubHeight_30, dtype: float64
count    43797.000000
mean         5.012410
std          2.179304
min          0.114327
25%          3.484709
50%          4.732100
75%          6.349425
max         17.411032
Name: WindSpeed_HubHeight_40, dtype: float64
count    43797.000000
mean         5.294002
std          2.288291
min          0.110660
25%          3.684065
50%   

Wind Veering (Directional Shear)

Wind direction often changes with altitude.

Large directional changes (wind veering) create uneven aerodynamic loading and turbulence, affecting turbine efficiency and mechanical stress.

In [21]:
df['Wind_Veering'] = (
    (df['winddirection_100m'] -
     df['winddirection_10m'] + 180) % 360
) - 180

print(df['Wind_Veering'].describe())


count    43800.000000
mean         4.465251
std         11.967561
min       -180.000000
25%          0.000000
50%          3.000000
75%          7.000000
max        179.000000
Name: Wind_Veering, dtype: float64


Wind Speed Components (U and V)

Wind direction is circular (0° and 360° represent the same direction). Converting wind into Cartesian components removes this discontinuity and allows machine learning models to interpret wind direction effectively.

In [23]:
# Convert degrees to radians
theta10 = np.radians(df['winddirection_10m'])
theta100 = np.radians(df['winddirection_100m'])

# 10 m components
df['U10'] = df['windspeed_10m'] * np.sin(theta10)
df['V10'] = df['windspeed_10m'] * np.cos(theta10)

# 100 m components
df['U100'] = df['windspeed_100m'] * np.sin(theta100)
df['V100'] = df['windspeed_100m'] * np.cos(theta100)

print(df[['U10', 'V10', 'U100', 'V100']].describe())

                U10           V10          U100          V100
count  43800.000000  43800.000000  43800.000000  43800.000000
mean      -1.172758     -0.262277     -2.037559     -0.463500
std        2.724392      2.598151      4.638308      4.563429
min      -13.284408    -10.304541    -20.395764    -16.990635
25%       -3.009075     -2.099798     -5.546356     -3.787279
50%       -1.305890     -0.384358     -2.393410     -0.413910
75%        0.917422      1.695259      1.609300      3.064436
max        8.208702      9.598538     13.890013     16.210119


Turbulence Intensity and Gust Factor

Turbulence Intensity (TI): This measures the degree of wind speed fluctuations.

High TI causes continuous blade pitching and yawing, which exacerbates material fatigue, reduces the operational lifespan of components, and lowers Annual Energy Production (AEP).

Gust Factor (GF): This highlights sudden, short-term wind spikes above the mean speed.

These sudden spikes introduce rapid mechanical and electrical power variations.

If gusts force rotor speeds beyond safety thresholds, they trigger emergency shutdowns to prevent catastrophic structural failure.

In [24]:
# Rolling statistics (6-hour window)
rolling_mean = df['windspeed_100m'].rolling(window=6).mean()
rolling_std = df['windspeed_100m'].rolling(window=6).std()

df['Turbulence_Intensity'] = rolling_std / rolling_mean

df['Gust_Factor'] = (
    df['windgusts_10m'] /
    df['windspeed_100m']
)
print(df[['Turbulence_Intensity', 'Gust_Factor']].describe())

       Turbulence_Intensity   Gust_Factor
count          43795.000000  43800.000000
mean               0.161667      1.336474
std                0.130705      0.804231
min                0.002216      0.275229
25%                0.073067      0.951296
50%                0.121463      1.192661
75%                0.205736      1.541904
max                1.149505     54.000000


Kinetic Energy Flux (Available Wind Power)

This feature estimates the theoretical energy available in the wind before turbine losses.

In [25]:
df['Kinetic_Energy_Flux'] = (
    0.5 *
    df['Air_Density'] *
    (df['windspeed_100m'] ** 3)
)
print(df['Kinetic_Energy_Flux'].describe())

count    43800.000000
mean       247.008628
std        318.155316
min          0.000589
25%         51.918546
50%        140.774878
75%        318.863622
max       5419.022065
Name: Kinetic_Energy_Flux, dtype: float64


Lag Features

Wind power generation has strong temporal autocorrelation.

The inclusion of previous observations helps models capture short-term variations and significantly improves forecasting accuracy.

In [27]:
# Power lags
df['Power_Lag_1'] = df['Power'].shift(1)
df['Power_Lag_2'] = df['Power'].shift(2)
df['Power_Lag_6'] = df['Power'].shift(6)

print(df[['Power_Lag_1', 'Power_Lag_2', 'Power_Lag_6']].describe())

# Wind speed lags
df['Wind100_Lag_1'] = df['windspeed_100m'].shift(1)
df['Wind100_Lag_2'] = df['windspeed_100m'].shift(2)
df['Wind100_Lag_6'] = df['windspeed_100m'].shift(6)

print(df[['Wind100_Lag_1', 'Wind100_Lag_2', 'Wind100_Lag_6']].describe())

        Power_Lag_1   Power_Lag_2   Power_Lag_6
count  43799.000000  43798.000000  43794.000000
mean       0.405388      0.405391      0.405410
std        0.288325      0.288328      0.288334
min        0.000000      0.000000      0.000000
25%        0.148900      0.148900      0.148900
50%        0.347700      0.347700      0.347700
75%        0.659600      0.659600      0.659675
max        0.991300      0.991300      0.991300
       Wind100_Lag_1  Wind100_Lag_2  Wind100_Lag_6
count   43799.000000   43798.000000   43794.000000
mean        6.284459       6.284497       6.284777
std         2.685240       2.685259       2.685190
min         0.100000       0.100000       0.100000
25%         4.380000       4.380000       4.380000
50%         6.080000       6.080000       6.080000
75%         7.990000       7.990000       7.990000
max        20.650000      20.650000      20.650000
